### [Neural Network on Trading Strategy](https://medium.com/@robinmgk077/i-trained-a-neural-network-on-a-terrible-trading-strategy-heres-what-it-still-learned-e427d693582b)

In [1]:
import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

In [2]:
##!kaggle datasets download jacksoncrow/stock-market-dataset --file stocks/AAPL.csv --unzip
!kaggle datasets download samanfatima7/2020-2025-apple-stock-dataset --unzip

Dataset URL: https://www.kaggle.com/datasets/samanfatima7/2020-2025-apple-stock-dataset
License(s): apache-2.0
  0% 0.00/51.3k [00:00<?, ?B/s]
100% 51.3k/51.3k [00:00<00:00, 188MB/s]


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.set_printoptions(precision=4, suppress=True)
pd.set_option('display.max_columns', None)
#pd.set_option('display.width', 90)
plt.style.use('ggplot')

import warnings
warnings.filterwarnings('ignore')

In [4]:
# - - - - - - - - - - - - - - -
# Load OHLC data
# Expected columns: ['open', 'high', 'low', 'close']
# - - - - - - - - - - - - - - -
df = pd.read_csv("/content/apple_5yr_one.csv")
# (Optional but recommended) enforce lowercase column names
df.columns = [c.strip().lower() for c in df.columns]
required_cols = {"open", "high", "low", "close"}

if not required_cols.issubset(set(df.columns)):
 raise ValueError(f"Missing columns. Expected at least: {required_cols}, got: {set(df.columns)}")

display(df.sample(8))

,date,close,high,low,open,volume
984,2024-05-01,168.28366088867188,171.67319360904702,168.0947990703537,168.56197877866052,50383100
1020,2024-06-24,207.17111206054688,211.70988292620214,205.62832424046638,206.75306897426384,80727000
703,2023-03-20,155.62442016601562,156.03969557063917,152.4110822679711,153.32071734921686,73641400
622,2022-11-18,149.35531616210938,150.74728887735557,148.0522039867333,150.36227675012384,74829600
141,2020-12-22,128.64923095703125,131.11725025660584,126.4738504005036,128.3858412033072,168904800
662,2023-01-19,133.54017639160156,134.50763996952077,132.05935833689125,132.3653916580124,58280400
1011,2024-06-10,192.2210235595703,196.3815736009198,191.25553768440525,195.98342648408087,97262100
302,2021-08-13,146.13072204589844,146.46394745377953,145.3172494643666,146.00330617321433,59375000


In [5]:
# - - - - - - - - - - - - - - -
# Strategy parameters (deliberately simple)
# - - - - - - - - - - - - - - -
FAST_MA = 10
SLOW_MA = 30
LOOKBACK = 50 # number of past candles the model sees
HORIZON = 20 # how far ahead we define "outcome"

In [6]:
print("Dataset Information:")
display(df.info())

print("\nDescriptive Statistics:")
display(df.describe())

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1256 entries, 0 to 1255
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   date    1255 non-null   object
 1   close   1256 non-null   object
 2   high    1256 non-null   object
 3   low     1256 non-null   object
 4   open    1256 non-null   object
 5   volume  1256 non-null   object
dtypes: object(6)
memory usage: 59.0+ KB


None


Descriptive Statistics:


,date,close,high,low,open,volume
count,1255,1256,1256,1256,1256,1256
unique,1255,1241,1256,1256,1256,1254
top,2025-06-02,223.69039916992188,202.1300048828125,200.1199951171875,200.27999877929688,97918500
freq,1,2,1,1,1,2


In [7]:
# - - - - - - - - - - - - - - -
# Moving averages and crossover detection
# - - - - - - - - - - - - - - -

# Convert OHLC columns to numeric, coercing errors
numeric_cols = ["open", "high", "low", "close"]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    else:
        raise ValueError(f"Required column '{col}' not found in DataFrame.")

# Impute NaN values in numeric columns with the mean of each column
for col in numeric_cols:
    if df[col].isnull().any():
        df[col].fillna(df[col].mean(), inplace=True)

# Reset index after potential changes, though not strictly needed after imputation
df.reset_index(drop=True, inplace=True)

df["fast_ma"] = df["close"].rolling(window=FAST_MA, min_periods=FAST_MA).mean()
df["slow_ma"] = df["close"].rolling(window=SLOW_MA, min_periods=SLOW_MA).mean()
# Signal: 1 when fast > slow, else 0
df["signal"] = (df["fast_ma"] > df["slow_ma"]).astype(int)
# Crossover: +1 = bullish crossover (fast crosses above slow)
df["crossover"] = df["signal"].diff()
# Entries: bullish crossovers only
entry_indices = df.index[df["crossover"] == 1].to_numpy()

display(df.sample(10))

,date,close,high,low,open,volume,fast_ma,slow_ma,signal,crossover
814,2023-08-28,178.644699,179.041263,177.018764,178.545551,43820700,175.668430,183.114446,0,0.0
830,2023-09-20,173.984985,178.158871,173.895746,177.722643,58436200,175.371996,177.643666,0,0.0
685,2023-02-22,147.230209,148.258471,145.499950,147.190652,51011300,150.222905,143.414874,1,0.0
429,2022-02-14,165.969528,166.657461,163.689504,164.485542,86185500,169.771275,167.738234,1,0.0
1084,2024-09-24,226.573547,228.546623,224.939293,227.849062,43556100,222.481947,223.378133,0,0.0
812,2023-08-24,174.867355,179.546877,174.500518,179.120557,54945800,175.514755,183.952570,0,0.0
501,2022-05-27,147.277206,147.316568,142.966361,143.094313,90978500,140.540268,150.594827,0,0.0
1167,2025-01-24,222.243881,225.087029,220.877183,224.239068,54697900,228.803064,242.367678,0,0.0
954,2024-03-19,175.022980,175.549797,171.991286,173.293420,55215200,171.104634,178.278555,0,0.0
495,2022-05-19,135.181244,139.423186,134.443086,137.671294,136095600,144.971178,155.674867,0,0.0


In [8]:
# - - - - - - - - - - - - - - -
# Build supervised dataset:
# X = [LOOKBACK x 4] OHLC window
# y = 1 if close(t + HORIZON) > close(t), else 0
# - - - - - - - - - - - - - - -
X, y = [], []
for idx in entry_indices:
 # Need enough history + enough future
 if idx < LOOKBACK:
  continue
 if idx + HORIZON >= len(df):
  continue
# Inputs: raw OHLC window ending right BEFORE the entry bar
 window = df.loc[idx - LOOKBACK : idx - 1, ["open", "high", "low", "close"]].values.astype(np.float32)
 X.append(window)
# Label: direction after fixed horizon (simple, honest, not TP/SL)
 future_return = float(df.loc[idx + HORIZON, "close"] - df.loc[idx, "close"])
 y.append(1 if future_return > 0 else 0)
X = np.asarray(X, dtype=np.float32)
y = np.asarray(y, dtype=np.float32)
print("X shape:", X.shape) # (samples, LOOKBACK, 4)
print("y shape:", y.shape) # (samples,)
print("Positive rate:", y.mean() if len(y) else "N/A")

X shape: (22, 50, 4)
y shape: (22,)
Positive rate: 0.59090906


In [9]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Safety checks
if X.ndim != 3 or X.shape[2] != 4:
 raise ValueError(f"Expected X shape (samples, time, 4). Got {X.shape}")

TIME_STEPS = X.shape[1] # LOOKBACK
FEATURES = X.shape[2] # 4 (OHLC)
model = models.Sequential([
 layers.Input(shape=(TIME_STEPS, FEATURES)),

 # Small LSTM: enough to model sequence structure, enough to overfit if it wants
 layers.LSTM(units=32, return_sequences=False),

 # Binary output: probability of success
 layers.Dense(1, activation="sigmoid")
])
model.compile(
 optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
 loss="binary_crossentropy",
 metrics=["accuracy"]
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 32)             │         4,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,769 (18.63 KB)

 Trainable params: 4,769 (18.63 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
# - - - - - - - - - - - - - - -
# Time-based split (no shuffling)
# - - - - - - - - - - - - - - -
n_samples = len(X)
if n_samples < 100:
 print("Warning: very small dataset. Results will be noisy.")
train_end = int(0.70 * n_samples)
val_end = int(0.85 * n_samples)
X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]
print("Train:", X_train.shape, y_train.shape)
print("Val: ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)

# - - - - - - - - - - - - - - -
# Train (deliberately basic)
# - - - - - - - - - - - - - - -
history = model.fit(
 X_train,
 y_train,
 validation_data=(X_val, y_val),
 epochs=10,
 batch_size=32,
 shuffle=False,
 verbose=1
)

Train: (15, 50, 4) (15,)
Val:  (3, 50, 4) (3,)
Test:  (4, 50, 4) (4,)
Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.4000 - loss: 0.9663 - val_accuracy: 0.3333 - val_loss: 1.0414
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.4000 - loss: 0.9585 - val_accuracy: 0.3333 - val_loss: 1.0299
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step - accuracy: 0.4000 - loss: 0.9424 - val_accuracy: 0.3333 - val_loss: 0.9981
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.4000 - loss: 0.9091 - val_accuracy: 0.3333 - val_loss: 0.9240
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 0.4000 - loss: 0.8577 - val_accuracy: 0.3333 - val_loss: 0.8525
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step - accuracy: 0.4000 - loss: 0.8111 - val_accuracy: 0.3333 - val_loss: 0.8266
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.4000 - loss: 0.7835 - val_accuracy: 0.3333 - val_loss: 0.8132
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accu

In [11]:
# Evaluate model on validation and test splits
val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Validation loss: {val_loss:.4f}")
print(f"Validation accuracy: {val_acc:.4f}")
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

# Majority-class baseline (computed from TRAIN labels only)
majority_class = 1.0 if (y_train.mean() >= 0.5) else 0.0
baseline_acc = float((y_test == majority_class).mean()) if len(y_test) else float("nan")
print(f"Majority baseline: {baseline_acc:.4f}")
print(f"Train positive rate: {y_train.mean():.4f}")
print(f"Test positive rate: {y_test.mean():.4f}")

Validation loss: 0.7373
Validation accuracy: 0.3333
Test loss: 0.7078
Test accuracy: 0.5000
Majority baseline: 0.5000
Train positive rate: 0.6000
Test positive rate: 0.5000


In [12]:
# Predicted probabilities on the test set
p_test = model.predict(X_test, verbose=0).reshape(-1)
print("Pred prob summary:")
print("min:", float(p_test.min()))
print("p25:", float(np.percentile(p_test, 25)))
print("p50:", float(np.percentile(p_test, 50)))
print("p75:", float(np.percentile(p_test, 75)))
print("max:", float(p_test.max()))
print("mean:", float(p_test.mean()))

# How often does it predict class 1 at the default threshold?
pred_1_rate = float((p_test >= 0.5).mean())
print("Predicted-1 rate (threshold=0.5):", pred_1_rate)

Pred prob summary:
min: 0.44032493233680725
p25: 0.4412008821964264
p50: 0.44275546073913574
p75: 0.446691632270813
max: 0.45471230149269104
mean: 0.44513705372810364
Predicted-1 rate (threshold=0.5): 0.0


In [13]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = (p_test >= 0.5).astype(int)
cm = confusion_matrix(y_test.astype(int), y_pred)
print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n")
print(classification_report(y_test.astype(int), y_pred, digits=4))

Confusion Matrix:
 [[2 0]
 [2 0]]

Classification Report:

              precision    recall  f1-score   support

           0     0.5000    1.0000    0.6667         2
           1     0.0000    0.0000    0.0000         2

    accuracy                         0.5000         4
   macro avg     0.2500    0.5000    0.3333         4
weighted avg     0.2500    0.5000    0.3333         4

